# Logistic Regression from Scratch: Gradient Descent on the Titanic Dataset

This notebook reimplements logistic regression example from Part 1, but instead of calling `LogisticRegression().fit()`,
we derive and implement **gradient descent** by hand.

We will:
1. Load and preprocess the Titanic dataset.
2. Review the mathematical foundations of logistic regression.
3. Implement gradient descent to learn the model weights.
4. Compare our results against scikit-learn.

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

## 1. Data Loading and Preprocessing

In [6]:
data = pd.read_csv("../data/titanic.csv")
data.head()

,sex,age,family_size,fare,1st_class,2nd_class,3rd_class,survived
0,1,22.0,1,7.2500,0,0,1,0
1,0,38.0,1,71.2833,1,0,0,1
2,0,26.0,0,7.9250,0,0,1,1
3,0,35.0,1,53.1000,1,0,0,1
4,1,35.0,0,8.0500,0,0,1,0


In [7]:
FEATURES = ["sex", "age", "family_size", "fare", "1st_class", "2nd_class", "3rd_class"]
TARGET = "survived"

features = data[FEATURES]

target = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

Training set: 709 samples, Test set: 178 samples


## 2. Feature Standardization

Gradient descent converges much faster when features are on a similar scale.
We standardize each feature to have zero mean and unit variance:

$$
x_j^{\prime} = \frac{x_j - \mu_j}{\sigma_j}
$$

We fit the statistics on the **training set only** and apply them to both sets
to avoid data leakage.

In [8]:
# TODO: Implement feature standardization.

"""
scaler = StandardScaler()

# Fit ONLY on training_dataset_generation data
X_train_standardized = scaler.fit_transform(X_train)

# Use same statistics on test data
X_test_standardized = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples, {X_train_scaled}")
print(f"Test set: {X_test.shape[0]} samples")
"""


numeric_features = ["age", "family_size", "fare"]
binary_features = ["sex", "1st_class", "2nd_class", "3rd_class"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
    ],
    remainder="passthrough",
)

x_train_standardized = preprocessor.fit_transform(X_train)
x_test_standardized = preprocessor.transform(X_test)

print(x_train_standardized, x_test_standardized)
"""

def train_standardizer(features_dataset):

    x_train_standardized = features_dataset.copy()
    for column in features_dataset.columns:
        mean = features_dataset[column].mean()
        std = features_dataset[column].std()

        print(f"Mean of {column}: {mean:.2f}, standard deviation of {column} {std:.2f}")

        x_train_standardized[column] = (features_dataset[column] - mean) / std


    return x_train_standardized

def test_standardizer(train_features_dataset, test_features_dataset):

    x_test_standardized = test_features_dataset.copy()
    for column in train_features_dataset.columns:
        mean = train_features_dataset[column].mean()
        std = train_features_dataset[column].std()

        print(f"Mean of {column}: {mean:.2f}, standard deviation of {column} {std:.2f}")

        x_test_standardized[column] = (test_features_dataset[column] - mean) / std


    return x_test_standardized

x_train_standardized = train_standardizer(X_train)
x_test_standardized = test_standardizer(X_train, X_test)

print(x_train_standardized, x_test_standardized)
"""

[[-0.44900956 -0.5539584  -0.38859731 ...  0.          1.
   0.        ]
 [-0.09415348  0.64529811 -0.48457737 ...  0.          0.
   1.        ]
 [-1.93940511  3.04381112 -0.04297452 ...  0.          0.
   1.        ]
 ...
 [-0.37803835 -0.5539584  -0.38859731 ...  0.          1.
   0.        ]
 [ 2.46081032  2.44418286  4.33948459 ...  1.          0.
   0.        ]
 [ 0.2607026  -0.5539584  -0.4707865  ...  0.          0.
   1.        ]] [[ 1.25429964 -0.5539584  -0.05763158 ...  1.          0.
   0.        ]
 [-0.30706713  1.24492636  0.15190188 ...  0.          1.
   0.        ]
 [ 2.8156664  -0.5539584  -0.36022882 ...  0.          0.
   1.        ]
 ...
 [-1.79746268  0.64529811 -0.3186217  ...  0.          0.
   1.        ]
 [-1.79746268  0.64529811  0.91367342 ...  1.          0.
   0.        ]
 [ 1.18332842 -0.5539584   0.1144706  ...  1.          0.
   0.        ]]


'\n\ndef train_standardizer(features_dataset):\n\n    x_train_standardized = features_dataset.copy()\n    for column in features_dataset.columns:\n        mean = features_dataset[column].mean()\n        std = features_dataset[column].std()\n\n        print(f"Mean of {column}: {mean:.2f}, standard deviation of {column} {std:.2f}")\n\n        x_train_standardized[column] = (features_dataset[column] - mean) / std\n\n\n    return x_train_standardized\n\ndef test_standardizer(train_features_dataset, test_features_dataset):\n\n    x_test_standardized = test_features_dataset.copy()\n    for column in train_features_dataset.columns:\n        mean = train_features_dataset[column].mean()\n        std = train_features_dataset[column].std()\n\n        print(f"Mean of {column}: {mean:.2f}, standard deviation of {column} {std:.2f}")\n\n        x_test_standardized[column] = (test_features_dataset[column] - mean) / std\n\n\n    return x_test_standardized\n\nx_train_standardized = train_standardizer(X_

## 3. Mathematical Background

### 3.1 The Logistic (Sigmoid) Function

In logistic regression we model the probability that a sample belongs to the
positive class ($y = 1$) using the **sigmoid function**:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

where the linear combination (logit) is:

$$
z = \mathbf{w}^\top \mathbf{x} + b = \sum_{j=1}^{d} w_j x_j + b
$$

Here $\mathbf{w} \in \mathbb{R}^d$ is the weight vector, $b \in \mathbb{R}$ is
the bias (intercept), and $d$ is the number of features.

The sigmoid function maps any real-valued input to a value between 0 and 1. 
We will use it to transform the model’s raw output (which can be any real number) into a probability. 
Large positive numbers are mapped close to 1, large negative numbers are mapped close to 0. 
This makes it suitable for binary classification.

The predicted probability is therefore:

$$
\hat{y} = P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b)
$$

The model computes a weighted sum of the input features and adds a bias term. 
This linear combination represents how strongly the input features support one class over the other. 
The sigmoid function then converts this value into a probability.

For example, a value of 0.8 means that the model estimated an 80% chance that the input belongs to the positive class.

### 3.2 Binary Cross-Entropy Loss

We measure how well the model fits the training data using the
**binary cross-entropy** (log loss), averaged over $n$ samples:

$$
\mathcal{L}(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n}
\left[
  y^{(i)} \ln\!\big(\hat{y}^{(i)}\big)
  + \big(1 - y^{(i)}\big) \ln\!\big(1 - \hat{y}^{(i)}\big)
\right]
$$

This loss is **convex** in $\mathbf{w}$ and $b$, so gradient descent is
guaranteed to find the global minimum.

The loss function measures how well the model predictions (the probabilities) match the true labels.  
Binary cross-entropy penalizes confident but incorrect predictions more heavily than uncertain ones.
If the model predicts a probability close to the true label, the loss is small, otherwise the loss increases rapidly.

### 3.3 Gradient Derivation

To compute how the loss changes with respect to the model parameters, gradients tell us in which direction we need to adjust the parameters to reduce the loss.

If the predictions are too high, this will lead to gradients with positive values. 
If predictions are too low, this will lead to gradients with negative values. 
The magnitude depends on how large the error is and the value of the input features.

To minimize $\mathcal{L}$ we need the partial derivatives with respect to each weight $w_j$ and the bias $b$.

Using the chain rule and the identity
$\sigma'(z) = \sigma(z)(1 - \sigma(z))$, the gradients simplify to:

$$
\frac{\partial \mathcal{L}}{\partial w_j}
= \frac{1}{n} \sum_{i=1}^{n}
  \big(\hat{y}^{(i)} - y^{(i)}\big)\, x_j^{(i)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b}
= \frac{1}{n} \sum_{i=1}^{n}
  \big(\hat{y}^{(i)} - y^{(i)}\big)
$$

Or in compact vector notation, with
$\hat{\mathbf{y}} = \sigma(X\mathbf{w} + b)$:

$$
\nabla_{\mathbf{w}} \mathcal{L}
= \frac{1}{n} X^\top (\hat{\mathbf{y}} - \mathbf{y})
\qquad\qquad
\frac{\partial \mathcal{L}}{\partial b}
= \frac{1}{n} \mathbf{1}^\top (\hat{\mathbf{y}} - \mathbf{y})
$$

### 3.4 Gradient Descent Update Rule

We iteratively update the parameters using a learning rate $\alpha$:

$$
\mathbf{w} \leftarrow \mathbf{w} - \alpha\, \nabla_{\mathbf{w}} \mathcal{L}
$$

$$
b \leftarrow b - \alpha\, \frac{\partial \mathcal{L}}{\partial b}
$$

At each step, the model updates the parameters slightly to reduce loss. Over many iterations, this leads to a set of parameters that fit the data well.

The learning rate $\alpha$ determines how large the update step is when adjusting the model parameters during training. 
Small learning rates (e.g. $\alpha = 0.001$) are slow but stable, whereas large learning rates ($\alpha = 0.1$) are fast, but may cause instabilities and training may fail completely.

We repeat until convergence (i.e., until the loss changes by less than a pre-defined tolerance $\varepsilon$), or if a maximum number of iterations is reached.

## 4. Implementation

Now it's time to implement the algorithm.
You need to account for numerical stability to ensure that the algorithm's returned results are sound.

In [10]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """A numerically stable sigmoid function."""
    # TODO: Implement this function.
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))


print(sigmoid(x_test_standardized))


def binary_cross_entropy(y: np.ndarray, y_hat: np.ndarray) -> float:
    """Compute the mean binary cross-entropy loss.

    Take care in your implementation to ensure that the cross entropy is always positive,
    and that it stays stable for very small probabilities (y_hat \approx 0).
    """
    # TODO: Implement this function.


def logistic_regression_gd(
    X: np.ndarray,
    y: np.ndarray,
    lr: float = 0.1,
    max_iter: int = 1000,
    tol: float = 1e-6,
) -> tuple[np.ndarray, float, list[float]]:
    """Train logistic regression via gradient descent.

    Returns (weights, bias, loss_history).
    """
    # TODO: Implement this function.

[[0.77804326 0.36494652 0.48559609 ... 0.73105858 0.5        0.5       ]
 [0.42383078 0.77642035 0.53790262 ... 0.5        0.73105858 0.5       ]
 [0.94351656 0.36494652 0.41090418 ... 0.5        0.5        0.73105858]
 ...
 [0.14216021 0.65595012 0.42101169 ... 0.5        0.5        0.73105858]
 [0.14216021 0.65595012 0.71375127 ... 0.73105858 0.5        0.5       ]
 [0.76554573 0.36494652 0.52858644 ... 0.73105858 0.5        0.5       ]]


## 5. Training

In [ ]:
# Train our model on the standardized features
w, b, loss_history = logistic_regression_gd(X_train_s, y_train, lr=0.1, max_iter=1000)

print(f"Final loss: {loss_history[-1]:.6f}")
print(f"Iterations: {len(loss_history)}")
print("\nLearned weights:")
for name, weight in zip(FEATURES, w):
    print(f"  {name:>30s}: {weight:+.4f}")
print(f"  {'bias':>30s}: {b:+.4f}")

### Loss Curve

A decreasing loss curve confirms that gradient descent is working correctly.

In [ ]:
# TODO: Implement a loss curve plot.

## 6. Evaluation

We classify a sample as positive ($\hat{y} = 1$) when $\sigma(z) \geq 0.5$,
which is equivalent to $z \geq 0$.

In [ ]:
def predict(X: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    """Predict class labels (0 or 1)."""
    # TODO: Implement this function.


def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute classification accuracy."""
    # TODO: Implement this function.


# Evaluate on train and test sets
y_pred_train = predict(X_train_s, w, b)
y_pred_test = predict(X_test_s, w, b)

print(f"Our model — Train accuracy: {accuracy(y_train, y_pred_train):.4f}")
print(f"Our model — Test accuracy:  {accuracy(y_test, y_pred_test):.4f}")

## 7. Comparison with scikit-learn

We train the same model using scikit-learn's `LogisticRegression` (with the
same standardized data) to verify that our gradient descent implementation
arrives at comparable accuracy and weights.

In [ ]:
# scikit-learn logistic regression (no regularization, to match our implementation)
sk_model = LogisticRegression(max_iter=200, random_state=42, C=np.inf)
sk_model.fit(X_train_s, y_train)

sk_accuracy = sk_model.score(X_test_s, y_test)
our_accuracy = accuracy(y_test, y_pred_test)

print(f"scikit-learn test accuracy: {sk_accuracy:.4f}")
print(f"Our GD test accuracy:      {our_accuracy:.4f}")
print()

# Compare learned weights
print(f"{'Feature':>30s} | {'Ours':>8s} | {'sklearn':>8s}")
print("-" * 55)
for name, w_ours, w_sk in zip(FEATURES, w, sk_model.coef_[0]):
    print(f"{name:>30s} | {w_ours:+8.4f} | {w_sk:+8.4f}")
print(f"{'bias':>30s} | {b:+8.4f} | {sk_model.intercept_[0]:+8.4f}")

## Summary

We implemented logistic regression from scratch using gradient descent and
verified it against scikit-learn. Key takeaways:

- The **sigmoid function** maps any real number to a probability in $(0, 1)$.
- The **binary cross-entropy loss** is convex, so gradient descent finds the
  global optimum.
- **Feature standardization** is critical for gradient descent to converge
  efficiently — without it, features on very different scales (e.g., Age vs.
  Fare) cause the loss surface to be poorly conditioned.
- Our hand-rolled implementation reaches the same accuracy and learns
  nearly identical weights as scikit-learn's optimized solver.